# OULAD corpus construction

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [1]:
import os
import sys
import hashlib
import pandas as pd

CANDIDATES = [
    BASE + '/raw',
    './data',
    '.',
]
RAW = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, 'studentInfo.csv'))), None)
if RAW is None:
    raise FileNotFoundError(
        "OULAD csvs not found. Put the 7 files in one of:\n  "
        + "\n  ".join(CANDIDATES)
        + "\n\nSource: Kuzilek, Hlosta & Zdrahal (2017), Scientific Data. CC-BY 4.0.\n"
          "Download page: https://analyse.kmi.open.ac.uk/open-dataset"
    )
print(f"OULAD raw directory: {os.path.abspath(RAW)}")

EXPECTED = {
    'studentInfo.csv':         32593,
    'studentRegistration.csv': 32593,
    'studentAssessment.csv':  173912,
    'studentVle.csv':       10655280,
    'assessments.csv':           206,
    'vle.csv':                  None,
    'courses.csv':               22,
}

oulad = {}
print(f"\n{'file':<26}{'rows':>10}{'expected':>10}  {'sha256[:8]':<12} status")

for name, exp in EXPECTED.items():
    path = os.path.join(RAW, name)
    if not os.path.exists(path):
        print(f"{name:<26}{'MISSING':>10}{'':>10}  {'':<12} *** NOT FOUND")
        continue

    df = pd.read_csv(path)
    oulad[name.replace('.csv', '')] = df

    with open(path, 'rb') as fh:
        digest = hashlib.sha256(fh.read()).hexdigest()[:8]

    if exp is None:
        status = 'no expectation set'
    elif len(df) == exp:
        status = 'OK'
    else:
        status = f'*** MISMATCH (off by {len(df) - exp:+,})'
    print(f"{name:<26}{len(df):>10,}{(exp if exp else '-'):>10}  {digest:<12} {status}")

bad = [n for n, e in EXPECTED.items() if e is not None
       and n.replace('.csv', '') in oulad and len(oulad[n.replace('.csv', '')]) != e]
assert not bad, (
    f"Row counts do not match the documented release for: {bad}\n"
    "Most likely cause: a duplicate upload was renamed (e.g. 'assessments (1).csv') and "
    "the original was read instead. Clear the runtime, fix the raw folder, and re-run."
)

studentInfo         = oulad['studentInfo']
studentRegistration = oulad['studentRegistration']
studentAssessment   = oulad['studentAssessment']
studentVle          = oulad['studentVle']
assessments         = oulad['assessments']
vle                 = oulad['vle']
courses             = oulad.get('courses')

OUT = os.path.dirname(RAW.rstrip('/')) if RAW.endswith('raw') else '.'
print(f"Derived artifacts will be read from / written to: {OUT}")

In [2]:
dupe_check = studentVle.groupby(['id_student', 'id_site', 'date']).size()
print(f"Total groups: {len(dupe_check)}, groups with >1 row: {(dupe_check > 1).sum()}")
print(dupe_check[dupe_check > 1].head(10))

Total groups: 8459320, groups with >1 row: 1614505
id_student  id_site  date
6516        877011   204     2
                     209     2
                     218     2
            877012   0       4
                     5       2
                     10      2
                     11      3
                     13      2
                     14      2
                     19      2
dtype: int64


In [3]:
studentVle_dedup = studentVle.groupby(
    ['code_module', 'code_presentation', 'id_student', 'id_site', 'date'],
    as_index=False
)['sum_click'].sum()

print(f"Before dedup: {len(studentVle):,} rows, after dedup: {len(studentVle_dedup):,} rows")
assert len(studentVle) == 10_655_280, 'studentVle row count drift -- check the raw folder'

merged = studentVle_dedup.merge(
    vle[['id_site', 'code_module', 'code_presentation', 'activity_type']],
    on=['id_site', 'code_module', 'code_presentation'], how='left'
)

click_by_type = merged.groupby('activity_type')['sum_click'].sum().sort_values(ascending=False)
pct = (click_by_type / click_by_type.sum() * 100).round(2)
print(pd.DataFrame({'total_clicks': click_by_type, 'pct_of_all_engagement': pct}))

oue_clicks = merged[merged['activity_type'] == 'ouelluminate'].groupby(
    ['code_module', 'code_presentation'])['sum_click'].sum()
print("\nouelluminate click volume by module/presentation:")
print(oue_clicks if len(oue_clicks) else "No ouelluminate clicks found in the dataset.")

Before dedup: 10,655,280 rows, after dedup: 8,459,320 rows
                total_clicks  pct_of_all_engagement
activity_type                                      
oucontent           11206803                  28.30
forumng              7973390                  20.13
quiz                 6981240                  17.63
homepage             6949064                  17.55
subpage              3411582                   8.61
resource             1110132                   2.80
ouwiki                894512                   2.26
url                   566702                   1.43
oucollaborate         108974                   0.28
glossary               87962                   0.22
questionnaire          64764                   0.16
externalquiz           64292                   0.16
page                   63631                   0.16
dataplus               47468                   0.12
ouelluminate           39028                   0.10
dualpane               20716                   0.05
htmla

In [4]:
print(len(studentVle_dedup), len(merged))

8459320 8459320


In [5]:
weight_check = assessments.groupby(['code_module','code_presentation'])['weight'].sum()
print("Weight sums per module-presentation (should be ~100):")
print(weight_check.describe())
print(f"\nModule-presentations where weight != 100: {(weight_check != 100).sum()} of {len(weight_check)}")

print("\nAssessment types and zero-weight counts:")
print(assessments.groupby('assessment_type')['weight'].agg(['count', lambda x: (x==0).sum()]))

print(f"\nTotal studentAssessment rows: {len(studentAssessment)}")
print(f"Rows with missing score: {studentAssessment['score'].isna().sum()}")
print(f"Rows with is_banked == 1 (credited from a prior attempt): {(studentAssessment['is_banked']==1).sum() if 'is_banked' in studentAssessment.columns else 'no is_banked column'}")

merged_assess = studentAssessment.merge(assessments[['id_assessment','code_module','code_presentation','assessment_type','date','weight']], on='id_assessment')
n_assess_per_student = merged_assess.groupby(['id_student','code_module','code_presentation']).size()
print("\nAssessments per student per presentation (need >=2 for a first-to-last trajectory/gain measure):")
print(n_assess_per_student.describe())
print(f"Students with only 1 assessment: {(n_assess_per_student==1).sum()} of {len(n_assess_per_student)}")

reg = studentRegistration[['id_student','code_module','code_presentation','date_unregistration']]
last_due = merged_assess.groupby(['code_module','code_presentation'])['date'].max().rename('last_assessment_due')
check = reg.merge(last_due, on=['code_module','code_presentation'], how='left')
withdrew_early = check[check['date_unregistration'] < check['last_assessment_due']]
print(f"\nStudents who withdrew before final assessment due date: {len(withdrew_early)} of {len(reg)}")
print(f"Students with no withdrawal date (completed/still active): {reg['date_unregistration'].isna().sum()}")

Weight sums per module-presentation (should be ~100):
count     22.000000
mean     195.454545
std       48.572702
min      100.000000
25%      200.000000
50%      200.000000
75%      200.000000
max      300.000000
Name: weight, dtype: float64

Module-presentations where weight != 100: 19 of 22

Assessment types and zero-weight counts:
                 count  <lambda_0>
assessment_type                   
CMA                 76          46
Exam                24           0
TMA                106          10

Total studentAssessment rows: 173912
Rows with missing score: 173
Rows with is_banked == 1 (credited from a prior attempt): 1909

Assessments per student per presentation (need >=2 for a first-to-last trajectory/gain measure):
count    25843.000000
mean         6.729559
std          3.771381
min          1.000000
25%          4.000000
50%          7.000000
75%         10.000000
max         14.000000
dtype: float64
Students with only 1 assessment: 2520 of 25843

Students who withdrew

In [6]:
valid = merged_assess[(merged_assess['is_banked'] == 0) & (merged_assess['weight'] > 0)]

n_valid = valid.groupby(['id_student','code_module','code_presentation']).size()
print(f"Students with >=2 valid assessments: {(n_valid >= 2).sum()} of {len(n_valid)}")

valid_sorted = valid.sort_values('date')
first_last = valid_sorted.groupby(['id_student','code_module','code_presentation']).agg(
    first_score=('score','first'), last_score=('score','last'),
    n_assessments=('score','size')
).reset_index()
first_last = first_last[first_last['n_assessments'] >= 2].copy()
first_last['performance_gain'] = first_last['last_score'] - first_last['first_score']

reg_small = studentRegistration[['id_student','code_module','code_presentation','date_unregistration']]
final = first_last.merge(reg_small, on=['id_student','code_module','code_presentation'], how='left')
last_due2 = valid.groupby(['code_module','code_presentation'])['date'].max().rename('last_valid_due')
final = final.merge(last_due2, on=['code_module','code_presentation'], how='left')
final = final[~(final['date_unregistration'] < final['last_valid_due'])]

print(f"\nFinal analytic sample size: {len(final)} of 32,593 registered students")
print(final['performance_gain'].describe())

clean_weight_modules = weight_check[weight_check == 100].index.tolist()
print(f"\nModule-presentations with weight sum == 100: {clean_weight_modules}")

Students with >=2 valid assessments: 20807 of 23239

Final analytic sample size: 18364 of 32,593 registered students
count    18363.000000
mean        -5.560148
std         20.345416
min        -95.000000
25%        -17.000000
50%         -4.000000
75%          7.000000
max         95.000000
Name: performance_gain, dtype: float64

Module-presentations with weight sum == 100: [('GGG', '2013J'), ('GGG', '2014B'), ('GGG', '2014J')]


In [7]:
final = final.dropna(subset=['first_score', 'last_score'])
print(f"After dropping NaN scores: {len(final)}")

After dropping NaN scores: 18363


In [8]:
last_types = valid_sorted.groupby(['id_student','code_module','code_presentation']).agg(
    first_type=('assessment_type','first'), last_type=('assessment_type','last')
)
last_types = last_types.loc[final.set_index(['id_student','code_module','code_presentation']).index.intersection(last_types.index)]
print("Last assessment type distribution:")
print(last_types['last_type'].value_counts())
print("\nFirst assessment type distribution:")
print(last_types['first_type'].value_counts())

final_with_types = final.merge(last_types.reset_index(), on=['id_student','code_module','code_presentation'])
print("\nMean performance_gain by last-assessment type:")
print(final_with_types.groupby('last_type')['performance_gain'].agg(['mean','std','count']))

Last assessment type distribution:
last_type
TMA     10558
Exam     4937
CMA      2868
Name: count, dtype: int64

First assessment type distribution:
first_type
TMA    15365
CMA     2998
Name: count, dtype: int64

Mean performance_gain by last-assessment type:
                mean        std  count
last_type                             
CMA         8.783821  22.084342   2868
Exam      -11.245088  19.140313   4937
TMA        -6.798257  18.538691  10558


In [9]:
print(final['performance_gain'].quantile([0.01, 0.05, 0.95, 0.99]))

0.01   -61.38
0.05   -42.00
0.95    26.00
0.99    40.00
Name: performance_gain, dtype: float64


In [10]:
lower, upper = final['performance_gain'].quantile([0.01, 0.99])
final['performance_gain_raw'] = final['performance_gain']
final['performance_gain'] = final['performance_gain'].clip(lower, upper)

n_capped = ((final['performance_gain_raw'] < lower) | (final['performance_gain_raw'] > upper)).sum()
print(f"Capped {n_capped} of {len(final)} students ({100*n_capped/len(final):.2f}%) at [{lower:.2f}, {upper:.2f}]")
print(final['performance_gain'].describe())

Capped 353 of 18363 students (1.92%) at [-61.38, 40.00]
count    18363.000000
mean        -5.543153
std         19.850827
min        -61.380000
25%        -17.000000
50%         -4.000000
75%          7.000000
max         40.000000
Name: performance_gain, dtype: float64


In [11]:
assert 'merged' in dir(), 'run the corpus-building cell first'

oucontent_engagement = merged[merged['activity_type'] == 'oucontent'].groupby(
    ['id_student', 'code_module', 'code_presentation']
)['sum_click'].sum().reset_index().rename(columns={'sum_click': 'oucontent_clicks'})

ouelluminate_engagement = merged[merged['activity_type'] == 'ouelluminate'].groupby(
    ['id_student', 'code_module', 'code_presentation']
)['sum_click'].sum().reset_index().rename(columns={'sum_click': 'ouelluminate_clicks'})

print(f"Students with any oucontent engagement: {len(oucontent_engagement)}")
print(f"Students with any ouelluminate engagement: {len(ouelluminate_engagement)}")

full_corpus = final.merge(
    oucontent_engagement, on=['id_student', 'code_module', 'code_presentation'], how='inner'
)

_n_before, _n_after = len(final), len(full_corpus)
print(f"\nF2 -- SELECTION ON T > 0")
print(f"  before inner join : {_n_before:,}")
print(f"  after inner join  : {_n_after:,}")
print(f"  dropped           : {_n_before - _n_after:,}     (expect 834)")
print(f"  min(oucontent_clicks) = {full_corpus['oucontent_clicks'].min()}"
      f"     (expect 1 -- confirms the criterion is T>0, not a threshold)")

assert _n_after == 17529, f"corpus size drift: {_n_after:,} (expect 17,529)"
assert full_corpus['oucontent_clicks'].min() >= 1, "T>0 criterion violated"

_mods = sorted(full_corpus['code_module'].unique())
print(f"\n  modules present : {_mods}")
assert len(_mods) == 6, f"expected 6 modules, found {len(_mods)}: {_mods}"

_expected = {'FFF': 4930, 'BBB': 4279, 'DDD': 3608, 'CCC': 2114, 'EEE': 1986, 'AAA': 612}
_actual = full_corpus['code_module'].value_counts().to_dict()
for _m, _e in _expected.items():
    _a = _actual.get(_m, 0)
    print(f"    {_m}  {_a:>5,}  expected {_e:>5,}  {'OK' if _a == _e else '*** MISMATCH'}")

oue_subsample = final.merge(
    ouelluminate_engagement, on=['id_student', 'code_module', 'code_presentation'], how='inner'
)
print(f"Ouelluminate subsample (multimedia treatment + valid outcome): {len(oue_subsample)}")
print(oue_subsample.groupby(['code_module', 'code_presentation']).size())

conf_cols = ['id_student', 'code_module', 'code_presentation', 'gender', 'region',
             'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
             'studied_credits', 'disability']

full_corpus = full_corpus.merge(studentInfo[conf_cols],
                                  on=['id_student', 'code_module', 'code_presentation'], how='left')
oue_subsample = oue_subsample.merge(studentInfo[conf_cols],
                                      on=['id_student', 'code_module', 'code_presentation'], how='left')

print("\nMissingness in full_corpus confounders (%):")
print((full_corpus[conf_cols].isna().mean() * 100).sort_values(ascending=False))

Students with any oucontent engagement: 26922
Students with any ouelluminate engagement: 2501

  before inner join : 18,363
  after inner join  : 17,529
  dropped           : 834     (expect 834)
  min(oucontent_clicks) = 1     (expect 1 -- confirms the criterion is T>0, not a threshold)

  modules present : ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF']

    FFF  4,930  expected 4,930  OK
    BBB  4,279  expected 4,279  OK
    DDD  3,608  expected 3,608  OK
    CCC  2,114  expected 2,114  OK
    EEE  1,986  expected 1,986  OK
    AAA    612  expected   612  OK
Ouelluminate subsample (multimedia treatment + valid outcome): 2031
code_module  code_presentation
BBB          2013B                390
DDD          2013B                649
FFF          2013B                992
dtype: int64

Missingness in full_corpus confounders (%):
imd_band                4.735011
code_module             0.000000
id_student              0.000000
code_presentation       0.000000
gender                  0.000000


In [12]:
print(full_corpus[full_corpus['imd_band'].isna()]['region'].value_counts())
print(full_corpus['region'].value_counts())

region
North Region            547
Ireland                 209
South Region             34
West Midlands Region     28
Scotland                  6
South West Region         3
North Western Region      2
Yorkshire Region          1
Name: count, dtype: int64
region
Scotland                2095
East Anglian Region     1780
South Region            1735
London Region           1564
North Western Region    1386
South West Region       1303
West Midlands Region    1262
Wales                   1206
East Midlands Region    1189
South East Region       1143
North Region            1049
Yorkshire Region        1002
Ireland                  815
Name: count, dtype: int64


In [13]:
import os
os.makedirs(OUT, exist_ok=True)

full_corpus.to_csv(f'{OUT}/oulad_full_corpus.csv', index=False)
oue_subsample.to_csv(f'{OUT}/oulad_ouelluminate_subsample.csv', index=False)

print(f"  Full corpus: {len(full_corpus)} rows")
print(f"  Ouelluminate subsample: {len(oue_subsample)} rows")

  Full corpus: 17529 rows
  Ouelluminate subsample: 2031 rows


In [15]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 120)

KEY = ['id_student', 'code_module', 'code_presentation']
CONF = ['gender', 'region', 'highest_education', 'imd_band', 'age_band',
        'num_of_prev_attempts', 'studied_credits', 'disability',
        'code_module', 'code_presentation']

def smd(a, b):
    """Standardised mean difference. |SMD| > 0.1 is the conventional flag."""
    sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return 0.0 if sp == 0 or np.isnan(sp) else (a.mean() - b.mean()) / sp

def balance(group_a, group_b, label_a, label_b):
    rows = []
    for c in CONF:
        if pd.api.types.is_numeric_dtype(group_a[c]):
            rows.append({'variable': c, 'level': '(numeric)',
                         label_a: group_a[c].mean(), label_b: group_b[c].mean(),
                         'SMD': smd(group_a[c].dropna(), group_b[c].dropna())})
        else:
            for lv in sorted(set(group_a[c].dropna()) | set(group_b[c].dropna())):
                rows.append({'variable': c, 'level': lv,
                             label_a: (group_a[c] == lv).mean(),
                             label_b: (group_b[c] == lv).mean(),
                             'SMD': smd((group_a[c] == lv).astype(float),
                                        (group_b[c] == lv).astype(float))})
    df = pd.DataFrame(rows)
    df['|SMD|'] = df['SMD'].abs()
    df['flag'] = np.where(df['|SMD|'] > 0.1, 'IMBALANCED', '')
    return df.sort_values('|SMD|', ascending=False).reset_index(drop=True)

si = studentInfo[KEY + [c for c in CONF if c not in KEY]].drop_duplicates(KEY)

final_k = final[KEY].drop_duplicates()
corpus_k = full_corpus[KEY].drop_duplicates()
flag = final_k.merge(corpus_k.assign(_kept=1), on=KEY, how='left')
dropped_k = flag[flag['_kept'].isna()][KEY]

print(f"in `final` (valid outcome)     : {len(final_k):,}")
print(f"in corpus (T>0 as well)        : {len(corpus_k):,}")
print(f"dropped by the inner join      : {len(dropped_k):,}")
assert len(dropped_k) == 834, f"expected 834, found {len(dropped_k):,}"

ret_834 = corpus_k.merge(si, on=KEY, how='left')
exc_834 = dropped_k.merge(si, on=KEY, how='left')

b1 = balance(ret_834, exc_834, 'retained', 'dropped_T0')
print(f"\nretained = {len(ret_834):,}   dropped at T>0 = {len(exc_834):,}\n")
print(b1.head(20).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\n{int((b1['|SMD|'] > 0.1).sum())} of {len(b1)} comparisons exceed |SMD| = 0.1")

print("\nmodule composition of the 834:")
comp = pd.DataFrame({
    'dropped_n': exc_834['code_module'].value_counts(),
    'dropped_pct': 100 * exc_834['code_module'].value_counts(normalize=True),
    'corpus_pct': 100 * ret_834['code_module'].value_counts(normalize=True)})
comp['ratio'] = comp['dropped_pct'] / comp['corpus_pct']
print(comp.sort_values('dropped_n', ascending=False).to_string(float_format=lambda v: f"{v:.2f}"))
print("\n  ratio > 1 = that module is OVER-represented among the T>0 exclusions.")

print("PART 2 — FULL ATTRITION CASCADE, EXCLUDING GGG STUDENTS")

si_noG = si[si['code_module'] != 'GGG']
kept_keys = set(map(tuple, corpus_k.values))
si_noG = si_noG.assign(_kept=[tuple(k) in kept_keys for k in si_noG[KEY].values])
ret_all, exc_all = si_noG[si_noG['_kept']], si_noG[~si_noG['_kept']]

b2 = balance(ret_all, exc_all, 'retained', 'excluded')
print(f"retained = {len(ret_all):,}   excluded = {len(exc_all):,}  (GGG removed from both)\n")
print(b2.head(20).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\n{int((b2['|SMD|'] > 0.1).sum())} of {len(b2)} comparisons exceed |SMD| = 0.1")

print("DISADVANTAGE GRADIENT — does it survive with GGG removed?")
markers = [('highest_education', 'Lower Than A Level'),
           ('highest_education', 'No Formal quals'),
           ('imd_band', '0-10%'),
           ('disability', 'Y'),
           ('gender', 'F'),
           ('num_of_prev_attempts', '(numeric)')]
for var, lv in markers:
    r = b2[(b2['variable'] == var) & (b2['level'] == lv)]
    if len(r):
        r = r.iloc[0]
        direction = 'UNDER-represented in corpus' if r['SMD'] < 0 else 'over-represented'
        print(f"  {var:<22} {lv:<24} SMD {r['SMD']:+.4f}   {direction}")


in `final` (valid outcome)     : 18,363
in corpus (T>0 as well)        : 17,529
dropped by the inner join      : 834

retained = 17,529   dropped at T>0 = 834

            variable                level  retained  dropped_T0       SMD    |SMD|       flag
         code_module                  BBB  0.244110    0.792566 -1.312687 1.312687 IMBALANCED
   code_presentation                2013B  0.146443    0.576739 -1.001217 1.001217 IMBALANCED
         code_module                  FFF  0.281248    0.000000  0.884623 0.884623 IMBALANCED
              gender                    M  0.596440    0.233813  0.791249 0.791249 IMBALANCED
              gender                    F  0.403560    0.766187 -0.791249 0.791249 IMBALANCED
         code_module                  DDD  0.205830    0.000000  0.719947 0.719947 IMBALANCED
   code_presentation                2014J  0.354384    0.135492  0.526206 0.526206 IMBALANCED
         code_module                  EEE  0.113298    0.000000  0.505504 0.505504 IMBAL